# LifeMate — ZIP file nhỏ của hynt + Sherpa để gửi review/inference

**CPU, không cần Internet/GPU/dataset audio.**

1. Add Input → Notebook Output của **hynt-finetune-continue** và **sherpa-finetune-continue**.
2. Đường dẫn đã điền theo hai báo cáo của bạn. Nếu mount khác, sửa `INPUT_DIRS` ở cell đầu;
   điền thư mục ngoài cùng cũng được, notebook tự tìm thư mục checkpoint con.
3. **Save Version → Save & Run All** (hoặc Run All trong editor).
4. Tải **`lifemate_finetune_handoff.zip`** ở link cuối hoặc mục Output/Files rồi gửi vào chat.

Một ZIP chứa hai thư mục `hynt/`, `sherpa_vi/`: config, runtime, tokenizer và source cần thiết.
Kèm metrics nhỏ và metadata để đối chiếu run/best nếu có. Có manifest SHA256 và kiểm tra CRC.
**Không chứa checkpoint `.pt`, audio, predictions, train/valid manifests hoặc log.**
`best.pt` vẫn ở Kaggle để notebook inference sau này Add Input, không cần tải về.
Best epoch trong manifest chỉ suy ra từ metrics; notebook không mở checkpoint để xác nhận weights.
Giữ notebook/output private.


In [ ]:
from pathlib import Path

INPUT_DIRS = {
    "hynt": "/kaggle/input/notebooks/minhdat27/hynt-finetune-continue/lifemate_hynt_continue_stage1_em",
    "sherpa_vi": "/kaggle/input/notebooks/minhdat27/sherpa-finetune-continue/lifemate_sherpa_vi_continue_stage1_em",
}
OUTPUT_DIR = Path("/kaggle/working/finetune_handoff")
MAX_BUNDLE_MIB = 32  # Dừng nếu bundle file nhỏ bất ngờ lớn hơn mức này.


In [ ]:
import hashlib
import json
import zipfile
from datetime import datetime, timezone

TRAINERS = {"hynt": "hynt_finetune.py", "sherpa_vi": "sherpa_vi_finetune.py"}
COMMON = ("train_config.json", "model_config.json", "runtime.json", "run_identity.json",
          "fbank_config.json", "bpe.model", "hynt_finetune_data.py")
OPTIONAL = ("state.json", "split_summary.json", "continuation_plan.json",
            "vocabulary_adaptation.json", "code_fix_resume.json")

def find_run(raw, trainer):
    if not str(raw).strip():
        raise ValueError("Input trống. Điền đường dẫn thư mục output đã Add Input.")
    root = Path(raw).expanduser()
    if not root.is_dir():
        raise FileNotFoundError(f"Chưa thấy {root}. Add Input output continuation tương ứng hoặc sửa INPUT_DIRS theo đường dẫn Kaggle.")
    root = root.resolve()
    if (root/trainer).is_file() and (root/'run_identity.json').is_file():
        return root
    matches = sorted({p.parent.resolve() for p in root.rglob('run_identity.json')
                      if (p.parent/trainer).is_file() and (p.parent/'best.pt').is_file()})
    if len(matches) != 1:
        raise ValueError(f"Cần đúng một run có {trainer}, run_identity.json và best.pt dưới {root}; tìm thấy {len(matches)}: {matches}. Điền đường dẫn thư mục run cụ thể.")
    return matches[0]

def create_bundle(input_dirs, output_dir, max_mib=32):
    if set(input_dirs) != set(TRAINERS):
        raise ValueError("INPUT_DIRS cần đúng hai nhãn hynt và sherpa_vi.")
    if not 0 < max_mib <= 256:
        raise ValueError("Giới hạn bundle phải trong (0,256] MiB.")
    plans = []
    run_info = {}
    for label, trainer in TRAINERS.items():
        root = find_run(input_dirs[label], trainer)
        required = COMMON+(trainer,)
        missing = [name for name in required if not (root/name).is_file()]
        if missing:
            raise FileNotFoundError(f"{label}: thiếu file cần thiết {missing} trong {root}")
        names = set(required) | {name for name in OPTIONAL if (root/name).is_file()}
        metrics = [p for p in root.glob('*_metrics.json') if p.name == 'baseline_metrics.json'
                   or p.name.startswith(('epoch_', 'partial_epoch_'))]
        names.update(p.name for p in metrics)
        candidates = []
        for path in metrics:
            item = json.loads(path.read_text(encoding='utf-8'))
            row = item.get('buckets', {}).get('all', {})
            if row.get('n', 0) and 'exact_normalized' in row and 'wer' in row:
                candidates.append((row['exact_normalized']/row['n'], -row['wer'],
                                   -item.get('global_step', 0), path.name, item.get('epoch')))
        identity = json.loads((root/'run_identity.json').read_text(encoding='utf-8'))
        run_info[label] = dict(source_directory=str(root), run_identity=identity.get('identity'),
            split_fingerprint=identity.get('split_fingerprint'),
            best_epoch_inferred_from_metrics=max(candidates)[4] if candidates else None,
            best_metrics_file_inferred=max(candidates)[3] if candidates else None,
            best_checkpoint_verified=False,
            best_checkpoint_path=str(root/'best.pt'),
            best_checkpoint_exists=(root/'best.pt').is_file(),
            note='Checkpoint weights are excluded and were not loaded. Keep this Kaggle input for inference.')
        print(f"\n{label}: {root}", flush=True)
        for name in sorted(names):
            path = root/name
            if path.is_symlink():
                raise ValueError(f"Không đóng gói symlink: {path}")
            size = path.stat().st_size
            print(f"  {name}: {size/1024:.1f} KiB")
            plans.append((label, path, size))
    total = sum(size for _, _, size in plans)
    if total > max_mib*1024**2:
        raise ValueError(f"File nhỏ tổng {total/1024**2:.2f} MiB vượt giới hạn {max_mib} MiB; kiểm tra input trước.")
    # All preflight checks finish before creating the archive.
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    target = output_dir/'lifemate_finetune_handoff.zip'
    partial = output_dir/'lifemate_finetune_handoff.zip.partial'
    if partial.exists():
        raise FileExistsError(f"Đang có {partial}; dùng OUTPUT_DIR mới nếu phiên trước bị ngắt.")
    manifest = dict(created_utc=datetime.now(timezone.utc).isoformat(), runs=run_info,
                    files=[], checkpoint_weights_included=False)
    try:
        with zipfile.ZipFile(partial, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
            for label, path, size in plans:
                payload = path.read_bytes()
                if len(payload) != size:
                    raise RuntimeError(f"Input đã thay đổi: {path}")
                name = label+'/'+path.name
                archive.writestr(name, payload)
                manifest['files'].append(dict(path=name, bytes=size, sha256=hashlib.sha256(payload).hexdigest()))
            archive.writestr('handoff_manifest.json', json.dumps(manifest, ensure_ascii=False, indent=2))
        with zipfile.ZipFile(partial) as archive:
            bad = archive.testzip()
            if bad is not None:
                raise RuntimeError(f"ZIP lỗi CRC: {bad}")
            for item in manifest['files']:
                if hashlib.sha256(archive.read(item['path'])).hexdigest() != item['sha256']:
                    raise RuntimeError('SHA256 không khớp: '+item['path'])
        partial.replace(target)
    except BaseException:
        partial.unlink(missing_ok=True)
        raise
    print(f"\nOK: {target} | {len(plans)} files | ZIP {target.stat().st_size/1024**2:.2f} MiB", flush=True)
    print("ZIP không chứa best.pt/last.pt. Tải ZIP này rồi gửi vào chat.")
    return target

ZIP_PATH = create_bundle(INPUT_DIRS, OUTPUT_DIR, MAX_BUNDLE_MIB)


In [ ]:
from IPython.display import FileLink, display
import os

display(FileLink(os.path.relpath(ZIP_PATH, Path.cwd())))
print("Nếu link không tải được, mở Output/Files → finetune_handoff → lifemate_finetune_handoff.zip")
